In [1]:
'''import pandas as pd
import numpy as np
import joblib
import os
import plotly.graph_objects as go
import plotly.express as px
import subprocess

# 0. SETUP DỮ LIỆU
path = '/kaggle/input/competitions/kkbox-churn-prediction-challenge/'

def extract_all_7z_to_csv(file_list):
    for file_7z in file_list:
        file_csv = file_7z.replace('.7z', '')
        if not os.path.exists(f'/kaggle/working/{file_csv}'):
            print(f"Đang bắt đầu giải nén: {file_7z} ...")
            try:
                subprocess.run(['7z', 'e', path + file_7z, '-o/kaggle/working/'], check=True)
                print(f"THÀNH CÔNG: Đã tạo file {file_csv}")
            except subprocess.CalledProcessError as e:
                print(f"LỖI khi xử lý {file_7z}: {e}")
        else:
            print(f"File {file_csv} đã tồn tại, bỏ qua giải nén.")

files_to_extract =[
    'train_v2.csv.7z',
    'members_v3.csv.7z',
    'transactions_v2.csv.7z',
    'user_logs_v2.csv.7z'
]

extract_all_7z_to_csv(files_to_extract)

INPUT_PATH = '/kaggle/working/'

print("\n--- ĐANG NẠP DỮ LIỆU THÔ ---")
train = pd.read_csv(f'{INPUT_PATH}train_v2.csv')
members = pd.read_csv(f'{INPUT_PATH}members_v3.csv')
transactions = pd.read_csv(f'{INPUT_PATH}transactions_v2.csv')

unique_users = train['msno'].unique()
logs = pd.read_csv(f'{INPUT_PATH}user_logs_v2.csv', nrows=5000000) 
logs = logs[logs['msno'].isin(unique_users)]

if not os.path.exists('/kaggle/input/notebooks/zile228/bi-project-predictive-analysis/kkbox_cox_model.pkl'):
    class MockModel:
        def __init__(self):
            self.params_ = {
                'Female': 0.01,
                'Age_15-20': 0.27,
                'Age_31-45': -0.06,
                'Price_Deal (<4.5d)': -0.15,
                'Pay_Manual': 0.48,
                'Skip_High (>50%)': 0.03
            }
    joblib.dump(MockModel(), 'kkbox_cox_model.pkl')
    print("-> Đã tạo file model giả lập để chạy test.")


# CLASS SIMULATION ENGINE 
class ChurnSimulationEngine:
    def __init__(self, data_path='streamlit_cohort_simulation.csv', model_path='kkbox_cox_model.pkl'):
        self.df = pd.read_csv(data_path)
        try:
            cph_model = joblib.load(model_path)
            self.coefs = cph_model.params_
        except:
            print("Cảnh báo: Không tìm thấy model Cox. Đang dùng hệ số mặc định.")
            self.coefs = {'Pay_Manual': 0.48, 'Price_Deal (<4.5d)': -0.15, 'Skip_High (>50%)': 0.03}
            
    # --- MODULE 1: WHAT-IF SIMULATION ---
    def run_what_if_simulation(self, filters=None, shift_auto=0.0, shift_upsell=0.0, shift_skip=0.0):
        df_sim = self.df.copy()
        if filters:
            for col, values in filters.items():
                if values:
                    df_sim = df_sim[df_sim[col].isin(values)]
                    
        if len(df_sim) == 0:
            return None, "Không có dữ liệu cho bộ lọc này."

        # FIX WARNING: Ép kiểu rõ ràng thành float ngay từ đầu
        df_sim['sim_revenue'] = df_sim['total_revenue'].astype(float)
        df_sim['sim_survival_days'] = df_sim['avg_survival_days'].astype(float)
        
        # Chiến dịch 1: Auto-Renew
        mask_manual = df_sim['payment_type'] == 'Pay_Manual'
        if shift_auto > 0 and mask_manual.any():
            shift_rate = shift_auto / 100.0
            impacted_users = df_sim.loc[mask_manual, 'user_count'] * shift_rate
            coef_manual = self.coefs.get('Pay_Manual', 0)
            new_hr = df_sim.loc[mask_manual, 'baseline_hr'] / np.exp(coef_manual)
            days_improvement_ratio = df_sim.loc[mask_manual, 'baseline_hr'] / new_hr
            new_days = df_sim.loc[mask_manual, 'avg_survival_days'] * days_improvement_ratio
            days_gained = (new_days - df_sim.loc[mask_manual, 'avg_survival_days'])
            revenue_gained = impacted_users * days_gained * df_sim.loc[mask_manual, 'avg_paid_per_day']
            df_sim.loc[mask_manual, 'sim_revenue'] += revenue_gained

        # Chiến dịch 2: Upsell
        mask_deal = df_sim['price_segment'] == 'Price_Deal (<4.5d)'
        if shift_upsell > 0 and mask_deal.any():
            shift_rate = shift_upsell / 100.0
            impacted_users = df_sim.loc[mask_deal, 'user_count'] * shift_rate
            coef_deal = self.coefs.get('Price_Deal (<4.5d)', 0)
            new_hr = df_sim.loc[mask_deal, 'baseline_hr'] / np.exp(coef_deal)
            days_improvement_ratio = df_sim.loc[mask_deal, 'baseline_hr'] / new_hr
            new_days = df_sim.loc[mask_deal, 'avg_survival_days'] * days_improvement_ratio
            rev_change = impacted_users * ( (new_days * 5.0) - (df_sim.loc[mask_deal, 'avg_survival_days'] * df_sim.loc[mask_deal, 'avg_paid_per_day']) )
            df_sim.loc[mask_deal, 'sim_revenue'] += rev_change

        # Chiến dịch 3: Tối ưu Suggestion
        mask_skip = df_sim['skip_segment'] == 'Skip_High (>50%)'
        if shift_skip > 0 and mask_skip.any():
            shift_rate = shift_skip / 100.0
            impacted_users = df_sim.loc[mask_skip, 'user_count'] * shift_rate
            coef_skip = self.coefs.get('Skip_High (>50%)', 0)
            new_hr = df_sim.loc[mask_skip, 'baseline_hr'] / np.exp(coef_skip)
            days_improvement_ratio = df_sim.loc[mask_skip, 'baseline_hr'] / new_hr
            new_days = df_sim.loc[mask_skip, 'avg_survival_days'] * days_improvement_ratio
            days_gained = (new_days - df_sim.loc[mask_skip, 'avg_survival_days'])
            revenue_gained = impacted_users * days_gained * df_sim.loc[mask_skip, 'avg_paid_per_day']
            df_sim.loc[mask_skip, 'sim_revenue'] += revenue_gained

        baseline_revenue = df_sim['total_revenue'].sum()
        total_new_revenue = df_sim['sim_revenue'].sum()
        revenue_delta = total_new_revenue - baseline_revenue
        
        kpis = {
            "Trạng thái gốc (Baseline)": baseline_revenue,
            "Tiền chênh lệch (Delta)": revenue_delta,
            "Kỳ vọng mới (Simulated)": total_new_revenue
        }
        
        fig = go.Figure(go.Waterfall(
            name="Revenue", orientation="v",
            measure=["absolute", "relative", "total"],
            x=["Doanh thu hiện tại", "Tác động giả lập", "Kỳ vọng tương lai"],
            y=[baseline_revenue, revenue_delta, total_new_revenue],
            connector={"line":{"color":"#BFC9CA"}},
            decreasing={"marker":{"color":"#E74C3C"}},
            increasing={"marker":{"color":"#2ECC71"}},
            totals={"marker":{"color":"#2980B9"}},
            text=[f"{v:,.0f} NTD" for v in[baseline_revenue, revenue_delta, total_new_revenue]],
            textposition="outside"
        ))
        fig.update_layout(title="Mô phỏng Tác động Chiến dịch (What-If Analysis)", plot_bgcolor='rgba(0,0,0,0)')

        return kpis, fig

    # --- MODULE 2: MONTE CARLO SIMULATION ---
    def run_monte_carlo(self, expected_revenue: float, volatility: float = 8.0, iterations: int = 10000):
        """
        Input: Doanh thu kỳ vọng từ What-If và độ biến động thị trường (%).
        """
        std_dev = expected_revenue * (volatility / 100)
        mc_results = np.random.normal(loc=expected_revenue, scale=std_dev, size=iterations)

        p10 = np.percentile(mc_results, 10)
        p50 = np.percentile(mc_results, 50)
        p90 = np.percentile(mc_results, 90)

        kpis = {"P10_worst": p10, "P50_expected": p50, "P90_best": p90}

        # Vẽ Histogram phân phối
        fig = px.histogram(
            mc_results, nbins=100, 
            title=f"Phân phối Doanh thu (Monte Carlo - Volatility: {volatility}%)",
            labels={'value': 'Doanh thu (NTD)'}, 
            color_discrete_sequence=['#143F6B']
        )
        # Thêm đường gióng P10, P90
        fig.add_vline(x=p10, line_dash="dash", line_color="red", annotation_text=f"Worst-case: P10")
        fig.add_vline(x=p90, line_dash="dash", line_color="green", annotation_text=f"Best-case: P90")

        return kpis, fig


# DEMO
print("\n=== TỔNG QUAN CHẠY MÔ PHỎNG DÀNH CHO C-LEVEL ===")
engine = ChurnSimulationEngine()

# 1. Chạy What-If
print("\n[1] ĐANG CHẠY WHAT-IF ANALYSIS...")
kpis_whatif, fig_whatif = engine.run_what_if_simulation(
    filters={'gender': ['Female']}, # Chỉ xem khách hàng Nữ
    shift_auto=30.0, 
    shift_upsell=0.0, 
    shift_skip=50.0
)
for k, v in kpis_whatif.items():
    print(f" - {k}: {v:,.0f} NTD")
fig_whatif.show()

# 2. Chạy Monte Carlo dựa trên kết quả What-If
print("\n[2] ĐANG CHẠY MONTE CARLO SIMULATION (Thị trường biến động 8%)...")
kpis_mc, fig_mc = engine.run_monte_carlo(
    expected_revenue=kpis_whatif["Kỳ vọng mới (Simulated)"], 
    volatility=8.0
)
for k, v in kpis_mc.items():
    print(f" - {k}: {v:,.0f} NTD")
fig_mc.show()'''

'import pandas as pd\nimport numpy as np\nimport joblib\nimport os\nimport plotly.graph_objects as go\nimport plotly.express as px\nimport subprocess\n\n# 0. SETUP DỮ LIỆU\npath = \'/kaggle/input/competitions/kkbox-churn-prediction-challenge/\'\n\ndef extract_all_7z_to_csv(file_list):\n    for file_7z in file_list:\n        file_csv = file_7z.replace(\'.7z\', \'\')\n        if not os.path.exists(f\'/kaggle/working/{file_csv}\'):\n            print(f"Đang bắt đầu giải nén: {file_7z} ...")\n            try:\n                subprocess.run([\'7z\', \'e\', path + file_7z, \'-o/kaggle/working/\'], check=True)\n                print(f"THÀNH CÔNG: Đã tạo file {file_csv}")\n            except subprocess.CalledProcessError as e:\n                print(f"LỖI khi xử lý {file_7z}: {e}")\n        else:\n            print(f"File {file_csv} đã tồn tại, bỏ qua giải nén.")\n\nfiles_to_extract =[\n    \'train_v2.csv.7z\',\n    \'members_v3.csv.7z\',\n    \'transactions_v2.csv.7z\',\n    \'user_logs_v2.c

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import plotly.graph_objects as go
import plotly.express as px
import subprocess

# ==============================================================================
# 0. SETUP DỮ LIỆU & MODEL GIẢ LẬP
# ==============================================================================
path = '/kaggle/input/competitions/kkbox-churn-prediction-challenge/'

def extract_all_7z_to_csv(file_list):
    for file_7z in file_list:
        file_csv = file_7z.replace('.7z', '')
        if not os.path.exists(f'/kaggle/working/{file_csv}'):
            print(f"Đang bắt đầu giải nén: {file_7z} ...")
            try:
                subprocess.run(['7z', 'e', path + file_7z, '-o/kaggle/working/'], check=True)
                print(f"THÀNH CÔNG: Đã tạo file {file_csv}")
            except subprocess.CalledProcessError as e:
                print(f"LỖI khi xử lý {file_7z}: {e}")
        else:
            print(f"File {file_csv} đã tồn tại, bỏ qua giải nén.")

files_to_extract =[
    'train_v2.csv.7z',
    'members_v3.csv.7z',
    'transactions_v2.csv.7z',
    'user_logs_v2.csv.7z'
]

extract_all_7z_to_csv(files_to_extract)

INPUT_PATH = '/kaggle/working/'

print("\n--- ĐANG NẠP DỮ LIỆU THÔ ---")
train = pd.read_csv(f'{INPUT_PATH}train_v2.csv')
members = pd.read_csv(f'{INPUT_PATH}members_v3.csv')
transactions = pd.read_csv(f'{INPUT_PATH}transactions_v2.csv')

unique_users = train['msno'].unique()
logs = pd.read_csv(f'{INPUT_PATH}user_logs_v2.csv', nrows=5000000) 
logs = logs[logs['msno'].isin(unique_users)]

# FIX LỖI PATH: Chỉ kiểm tra file ở ngay thư mục hiện hành
model_name = 'kkbox_cox_model.pkl'
if not os.path.exists(model_name):
    class MockModel:
        def __init__(self):
            self.params_ = {
                'Female': 0.01,
                'Age_15-20': 0.27,
                'Age_31-45': -0.06,
                'Price_Deal (<4.5d)': -0.15,
                'Pay_Manual': 0.48,
                'Skip_High (>50%)': 0.03
            }
    joblib.dump(MockModel(), model_name)
    print("-> Đã tự động tạo file model giả lập (Mock Model) để chạy test.")
else:
    print(f"-> Đã tìm thấy file model {model_name} sẵn có.")


# ==============================================================================
# 1. HÀM BUILD DATASET (ĐÃ FIX LỖI PANDAS DATETIME)
# ==============================================================================
def build_cohort_simulation_dataset(transactions, logs, members, model_path='kkbox_cox_model.pkl'):
    print("\n--- BẮT ĐẦU BUILD COHORT DATASET ---")
    cph_model = joblib.load(model_path)
    coefs = cph_model.params_
    CUT_OFF_DATE = pd.to_datetime('2017-03-31')
    
    trans_valid = transactions.copy()
    trans_valid['transaction_date'] = pd.to_datetime(trans_valid['transaction_date'], format='%Y%m%d')
    trans_valid['membership_expire_date'] = pd.to_datetime(trans_valid['membership_expire_date'], format='%Y%m%d')
    
    trans_agg = trans_valid.groupby('msno').agg(
        total_revenue=('actual_amount_paid', 'sum'),
        total_plan_days=('payment_plan_days', 'sum'),
        auto_renew_ratio=('is_auto_renew', 'mean'),
        first_date=('transaction_date', 'min'),
        last_date=('membership_expire_date', 'max') 
    ).reset_index()

    trans_agg['is_censored'] = trans_agg['last_date'] > CUT_OFF_DATE
    
    # FIX LỖI DATETIME Ở ĐÂY: Bọc kết quả np.where bằng pd.to_datetime
    trans_agg['end_calc_date'] = pd.to_datetime(
        np.where(trans_agg['is_censored'], CUT_OFF_DATE, trans_agg['last_date'])
    )
    
    trans_agg['survival_days'] = (trans_agg['end_calc_date'] - trans_agg['first_date']).dt.days.clip(lower=1)
    
    trans_agg['paid_per_day'] = np.where(trans_agg['total_plan_days'] > 0, 
                                         trans_agg['total_revenue'] / trans_agg['total_plan_days'], 0)
    trans_agg['payment_type'] = np.where(trans_agg['auto_renew_ratio'] > 0.5, 'Pay_Auto-Renew', 'Pay_Manual')
    trans_agg['price_segment'] = pd.cut(trans_agg['paid_per_day'], bins=[-0.1, 0.1, 4.5, float('inf')],
                                        labels=['Price_0d (Trial)', 'Price_Deal (<4.5d)', 'Price_Chuan (>=4.5d)'])

    logs['total_songs'] = logs['num_25'] + logs['num_50'] + logs['num_75'] + logs['num_985'] + logs['num_100']
    logs_agg = logs.groupby('msno').agg(
        total_skip=('num_25', 'sum'),
        total_songs=('total_songs', 'sum')
    ).reset_index()
    logs_agg['skip_ratio'] = np.where(logs_agg['total_songs'] > 0, logs_agg['total_skip'] / logs_agg['total_songs'], 0)
    logs_agg['skip_segment'] = pd.cut(logs_agg['skip_ratio'], bins=[-0.1, 0.2, 0.5, float('inf')],
                                      labels=['Skip_Low (<20%)', 'Skip_Med (20-50%)', 'Skip_High (>50%)'])

    members_lean = members[['msno', 'city', 'gender', 'bd']].copy()
    members_lean['gender'] = members_lean['gender'].fillna('Gender_Unknown')
    members_lean['gender'] = members_lean['gender'].replace({'male':'Male', 'female': 'Female'})
    members_lean['age_segment'] = pd.cut(members_lean['bd'], bins=[14, 20, 30, 45, 65], 
                                         labels=['Age_15-20', 'Age_21-30', 'Age_31-45', 'Age_46-65'])
    members_lean['age_segment'] = members_lean['age_segment'].astype(str).replace('nan', 'Age_Unknown')

    master = trans_agg.merge(logs_agg[['msno', 'skip_segment']], on='msno', how='inner')
    master = master.merge(members_lean, on='msno', how='inner')

    dimensions =['city', 'gender', 'age_segment', 'payment_type', 'price_segment', 'skip_segment']
    cohort_df = master.groupby(dimensions, observed=True).agg(
        user_count=('msno', 'count'),
        total_revenue=('total_revenue', 'sum'),
        avg_survival_days=('survival_days', 'mean'),
        avg_paid_per_day=('paid_per_day', 'mean')
    ).reset_index()
    
    def calc_cohort_hr(row):
        log_hr = 0.0
        if row['gender'] in coefs: log_hr += coefs[row['gender']]
        if row['age_segment'] in coefs: log_hr += coefs[row['age_segment']]
        if row['payment_type'] in coefs: log_hr += coefs[row['payment_type']]
        if row['price_segment'] in coefs: log_hr += coefs[row['price_segment']]
        if row['skip_segment'] in coefs: log_hr += coefs[row['skip_segment']]
        return np.exp(log_hr)

    cohort_df['baseline_hr'] = cohort_df.apply(calc_cohort_hr, axis=1)
    cohort_df.to_csv('streamlit_cohort_simulation.csv', index=False)
    print(f"Hoàn tất Build! File tạo ra có: {len(cohort_df):,} cohorts.")
    return cohort_df

cohort_data = build_cohort_simulation_dataset(transactions, logs, members)


# ==============================================================================
# 2. CLASS SIMULATION ENGINE (FULL PRD + MONTE CARLO)
# ==============================================================================
class ChurnSimulationEngine:
    def __init__(self, data_path='streamlit_cohort_simulation.csv', model_path='kkbox_cox_model.pkl'):
        self.df = pd.read_csv(data_path)
        try:
            cph_model = joblib.load(model_path)
            self.coefs = cph_model.params_
        except:
            self.coefs = {'Pay_Manual': 0.48, 'Price_Deal (<4.5d)': -0.15, 'Skip_High (>50%)': 0.03}
            
        self.colors = {
            'baseline': '#95A5A6',   
            'simulated': '#1ABC9C',  
            'primary': '#2C3E50',    
            'upsell': '#3498DB',     
            'negative': '#E74C3C'    
        }

    def _calculate_scenario(self, df_input, shift_auto=0.0, shift_upsell=0.0, shift_skip=0.0):
        df_sim = df_input.copy()
        
        retention_revenue = 0.0
        upsell_revenue = 0.0
        hr_distribution =[]
        
        for idx, row in df_sim.iterrows():
            users = row['user_count']
            base_hr = row['baseline_hr']
            base_days = row['avg_survival_days']
            base_price = row['avg_paid_per_day']
            
            unchanged_users = users
            
            if shift_auto > 0 and row['payment_type'] == 'Pay_Manual':
                rate = shift_auto / 100.0
                shifted_users = users * rate
                unchanged_users -= shifted_users
                
                coef = self.coefs.get('Pay_Manual', 0)
                new_hr = base_hr / np.exp(coef) 
                new_days = base_days * (base_hr / new_hr)
                
                retention_revenue += shifted_users * (new_days - base_days) * base_price
                hr_distribution.append({'type': 'Simulated (Tương lai)', 'hr': new_hr, 'users': shifted_users})

            elif shift_upsell > 0 and row['price_segment'] == 'Price_Deal (<4.5d)':
                rate = shift_upsell / 100.0
                shifted_users = users * rate
                unchanged_users -= shifted_users
                
                coef = self.coefs.get('Price_Deal (<4.5d)', 0)
                new_hr = base_hr / np.exp(coef) 
                new_days = base_days * (base_hr / new_hr)
                
                retention_revenue += shifted_users * (new_days - base_days) * base_price
                target_price = 5.0 
                upsell_revenue += shifted_users * new_days * (target_price - base_price)
                
                hr_distribution.append({'type': 'Simulated (Tương lai)', 'hr': new_hr, 'users': shifted_users})

            elif shift_skip > 0 and row['skip_segment'] == 'Skip_High (>50%)':
                rate = shift_skip / 100.0
                shifted_users = users * rate
                unchanged_users -= shifted_users
                
                coef = self.coefs.get('Skip_High (>50%)', 0)
                new_hr = base_hr / np.exp(coef)
                new_days = base_days * (base_hr / new_hr)
                
                retention_revenue += shifted_users * (new_days - base_days) * base_price
                hr_distribution.append({'type': 'Simulated (Tương lai)', 'hr': new_hr, 'users': shifted_users})
            
            hr_distribution.append({'type': 'Baseline (Hiện tại)', 'hr': base_hr, 'users': users})
            if unchanged_users > 0:
                hr_distribution.append({'type': 'Simulated (Tương lai)', 'hr': base_hr, 'users': unchanged_users})

        baseline_revenue = df_input['total_revenue'].sum()
        total_projected = baseline_revenue + retention_revenue + upsell_revenue
        
        return {
            'baseline_rev': baseline_revenue,
            'retention_rev': retention_revenue,
            'upsell_rev': upsell_revenue,
            'projected_rev': total_projected,
            'hr_dist_df': pd.DataFrame(hr_distribution)
        }

    def plot_hazard_shift(self, hr_dist_df):
        hr_dist_df['hr_bin'] = hr_dist_df['hr'].round(2)
        agg_dist = hr_dist_df.groupby(['type', 'hr_bin'])['users'].sum().reset_index()
        
        total_users = agg_dist[agg_dist['type'] == 'Baseline (Hiện tại)']['users'].sum()
        agg_dist['user_pct'] = (agg_dist['users'] / total_users) * 100

        fig = px.area(
            agg_dist, x='hr_bin', y='user_pct', color='type',
            color_discrete_map={'Baseline (Hiện tại)': self.colors['baseline'], 'Simulated (Tương lai)': self.colors['simulated']},
            labels={'hr_bin': 'Hệ số Rủi ro rời bỏ (Hazard Ratio)', 'user_pct': 'Tỷ trọng Khách hàng (%)'},
            title='Biểu đồ 1: Dịch chuyển Cấu trúc Rủi ro Hệ thống (Population Hazard Shift)'
        )
        fig.update_traces(opacity=0.6, line=dict(width=2))
        fig.update_layout(
            plot_bgcolor='white', hovermode='x unified',
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#F2F3F4')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#F2F3F4')
        return fig

    def plot_waterfall(self, res):
        fig = go.Figure(go.Waterfall(
            name="Revenue", orientation="v",
            measure=["absolute", "relative", "relative", "total"],
            x=["Doanh thu Hiện tại<br>(Baseline)", "Cứu vãn nhờ Giữ chân<br>(Retention)", 
               "Tăng thêm nhờ Ép giá<br>(Upsell)", "Kỳ vọng Tương lai<br>(Projected)"],
            y=[res['baseline_rev'], res['retention_rev'], res['upsell_rev'], res['projected_rev']],
            connector={"line":{"color":"#BDC3C7", "width": 2, "dash": "dot"}},
            decreasing={"marker":{"color": self.colors['negative']}},
            increasing={"marker":{"color": self.colors['simulated']}},
            totals={"marker":{"color": self.colors['primary']}},
            text=[f"{v/1e6:,.1f}M" for v in [res['baseline_rev'], res['retention_rev'], res['upsell_rev'], res['projected_rev']]],
            textposition="outside",
            textfont=dict(size=13, family="Arial", color="black")
        ))
        
        fig.update_layout(
            title='Biểu đồ 2: Phân rã Tác động Tài chính (Financial Impact Analysis)', 
            plot_bgcolor='white',
            yaxis=dict(title="Doanh thu (NTD)"),
            margin=dict(t=70, l=50, r=50, b=50)
        )
        return fig

    def plot_sensitivity(self, df_input):
        res_auto = self._calculate_scenario(df_input, shift_auto=1.0)
        res_upsell = self._calculate_scenario(df_input, shift_upsell=1.0)
        res_skip = self._calculate_scenario(df_input, shift_skip=1.0)
        
        roi_auto = res_auto['projected_rev'] - res_auto['baseline_rev']
        roi_upsell = res_upsell['projected_rev'] - res_upsell['baseline_rev']
        roi_skip = res_skip['projected_rev'] - res_skip['baseline_rev']
        
        data = pd.DataFrame({
            'Chiến lược':['Thuyết phục Auto-Renew', 'Giảm lượng Skip (Sản phẩm)', 'Ép gói Giá chuẩn (Upsell)'],
            'ROI_per_1_percent':[roi_auto, roi_skip, roi_upsell]
        }).sort_values('ROI_per_1_percent', ascending=True) 

        fig = px.bar(
            data, x='ROI_per_1_percent', y='Chiến lược', orientation='h',
            text_auto='.0f', color='ROI_per_1_percent', color_continuous_scale='Teal',
            title='Biểu đồ 3: Phân tích Độ nhạy (Giá trị Doanh thu mang lại từ mỗi 1% Nỗ lực)'
        )
        fig.update_layout(
            plot_bgcolor='white', coloraxis_showscale=False,
            xaxis=dict(title="Lợi nhuận tăng thêm trên mỗi 1% Chuyển đổi (NTD)"),
            yaxis=dict(title=""),
            font=dict(size=14)
        )
        fig.update_traces(texttemplate='%{x:,.0f} NTD', textposition='outside')
        return fig

    def run_monte_carlo(self, expected_revenue: float, volatility: float = 8.0, iterations: int = 10000):
        std_dev = expected_revenue * (volatility / 100)
        mc_results = np.random.normal(loc=expected_revenue, scale=std_dev, size=iterations)

        p10 = np.percentile(mc_results, 10)
        p50 = np.percentile(mc_results, 50)
        p90 = np.percentile(mc_results, 90)

        df_mc = pd.DataFrame({'Doanh thu': mc_results})

        fig = px.histogram(
            df_mc, x='Doanh thu', nbins=100, 
            title=f'Biểu đồ 4: Phân phối Rủi ro Doanh thu - Monte Carlo (Độ biến động: {volatility}%)',
            labels={'Doanh thu': 'Doanh thu Kỳ vọng (NTD)'}, 
            color_discrete_sequence=[self.colors['primary']]
        )
        
        fig.update_layout(
            showlegend=False, 
            plot_bgcolor='white', 
            yaxis=dict(title="Tần suất (Số Kịch bản)")
        )

        fig.add_vline(x=p10, line_dash="dash", line_color=self.colors['negative'], 
                      annotation_text=f"Trường hợp Xấu (P10): {p10/1e6:.1f}M")
        fig.add_vline(x=p90, line_dash="dash", line_color=self.colors['simulated'], 
                      annotation_text=f"Trường hợp Tốt (P90): {p90/1e6:.1f}M")
        
        return {"P10_worst": p10, "P50_expected": p50, "P90_best": p90}, fig

    def run_dashboard(self, filters=None, shift_auto=0.0, shift_upsell=0.0, shift_skip=0.0):
        df_sim = self.df.copy()
        if filters:
            for col, values in filters.items():
                if values:
                    df_sim = df_sim[df_sim[col].isin(values)]
                    
        if len(df_sim) == 0:
            print("Không có dữ liệu cho bộ lọc này.")
            return None, None, None, None

        res = self._calculate_scenario(df_sim, shift_auto, shift_upsell, shift_skip)
        
        fig_hazard = self.plot_hazard_shift(res['hr_dist_df'])
        fig_waterfall = self.plot_waterfall(res)
        fig_sensitivity = self.plot_sensitivity(df_sim)
        
        return res, fig_hazard, fig_waterfall, fig_sensitivity


# ==============================================================================
# 3. THỰC THI TRÊN KAGGLE VÀ HIỂN THỊ KẾT QUẢ
# ==============================================================================
print("\n" + "="*60)
print(" KHỞI ĐỘNG HỆ THỐNG MÔ PHỎNG CHIẾN LƯỢC K-CID (TAB 3)")
print("="*60)

engine = ChurnSimulationEngine()

# CHỈ ĐỊNH THAM SỐ TỪ C-LEVEL
FILTERS = {'gender': ['Female']}
SHIFT_AUTO = 40.0   
SHIFT_UPSELL = 30.0 
SHIFT_SKIP = 15.0   

print("\n[1] ĐANG TÍNH TOÁN CÁC BIỂU ĐỒ PRD CHÍNH...")
res_dict, fig_1, fig_2, fig_3 = engine.run_dashboard(
    filters=FILTERS, 
    shift_auto=SHIFT_AUTO, 
    shift_upsell=SHIFT_UPSELL, 
    shift_skip=SHIFT_SKIP
)

print(f" -> Tóm tắt Nhanh:")
print(f"    - Doanh thu Gốc: {res_dict['baseline_rev']:,.0f} NTD")
print(f"    - Tiền Giữ chân (Retention): {res_dict['retention_rev']:,.0f} NTD")
print(f"    - Tiền Tăng thêm (Upsell): {res_dict['upsell_rev']:,.0f} NTD")
print(f"    - KỲ VỌNG MỚI: {res_dict['projected_rev']:,.0f} NTD")

fig_1.show()
fig_2.show()
fig_3.show()

print("\n[2] ĐANG CHẠY 10,000 KỊCH BẢN MONTE CARLO (CFO VIEW)...")
mc_kpis, fig_4 = engine.run_monte_carlo(expected_revenue=res_dict['projected_rev'], volatility=8.0)

print(f" -> P10 (Xấu nhất): {mc_kpis['P10_worst']:,.0f} NTD")
print(f" -> P50 (Kỳ vọng):  {mc_kpis['P50_expected']:,.0f} NTD")
print(f" -> P90 (Tốt nhất): {mc_kpis['P90_best']:,.0f} NTD")

fig_4.show()

Đang bắt đầu giải nén: train_v2.csv.7z ...

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,4 CPUs AMD EPYC 7B12 (830F10),ASM,AES-NI)

Scanning the drive for archives:
1 file, 32818991 bytes (32 MiB)

Extracting archive: /kaggle/input/competitions/kkbox-churn-prediction-challenge/train_v2.csv.7z
--
Path = /kaggle/input/competitions/kkbox-churn-prediction-challenge/train_v2.csv.7z
Type = 7z
Physical Size = 32818991
Headers Size = 178
Method = LZMA2:24
Solid = -
Blocks = 1

Everything is Ok

Size:       45635134
Compressed: 32818991
THÀNH CÔNG: Đã tạo file train_v2.csv
Đang bắt đầu giải nén: members_v3.csv.7z ...

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,4 CPUs AMD EPYC 7B12 (830F10),ASM,AES-NI)

Scanning the drive for archives:
1 file, 242308558 bytes (232 MiB)

Extracting archive: /kaggle/input/competi


[2] ĐANG CHẠY 10,000 KỊCH BẢN MONTE CARLO (CFO VIEW)...
 -> P10 (Xấu nhất): 32,291,151 NTD
 -> P50 (Kỳ vọng):  36,045,778 NTD
 -> P90 (Tốt nhất): 39,660,670 NTD
